# Evaluate the Exported Kaldi Model

Computes raw (non-normalized) WER/CER for `train_kaldi.ipynb`'s trained
SAT (`tri3`) model on its held-out test set, and exports a per-utterance
results table for `compare.ipynb`.

**Runs on Windows, not WSL.** Kaldi itself doesn't build on native
Windows, so the actual decoding (inference) happens in `train_kaldi.ipynb`
under WSL; this notebook loads that already-decoded output (the exported
model's predictions) and scores it -- no Kaldi binaries are invoked here,
only plain-text parsing and `jiwer`.

**Prerequisite:** `train_kaldi.ipynb` must have run through its decode and
export stages, so `RESULTS_DIR/tri3/decode_test/` exists.

**Environment:** plain Python. Needs `pandas`, `pyarrow`, `jiwer`.


## 1. Load Exported Model Output

In [1]:
import os
import re
from pathlib import Path

import pandas as pd

RESULTS_DIR = Path(os.environ.get("BISAYA_RESULTS_DIR", "output")).resolve()
CORPUS_DIR = Path(os.environ.get("BISAYA_CORPUS_DIR", "data/bisaya_audio")).resolve()

DECODE_DIR = RESULTS_DIR / "tri3" / "decode_test"
SCORING_DIR = DECODE_DIR / "scoring_kaldi"

assert SCORING_DIR.exists(), (
    f"{SCORING_DIR} not found -- run train_kaldi.ipynb's decode + export "
    f"stages first."
)

# Kaldi's own scoring sweeps a range of LM weights/word-insertion
# penalties; best_wer records the winning combination.
best_wer_line = (SCORING_DIR / "best_wer").read_text(encoding="utf-8").strip()
m = re.search(r"wer_(\d+)_([\d.]+)\s*$", best_wer_line)
assert m, f"could not parse LM weight / word-insertion penalty from: {best_wer_line}"
BEST_LMWT, BEST_WIP = m.group(1), m.group(2)

HYP_PATH = SCORING_DIR / f"penalty_{BEST_WIP}" / f"{BEST_LMWT}.txt"
REF_PATH = SCORING_DIR / "test_filt.txt"
assert HYP_PATH.exists() and REF_PATH.exists()

print("Best result:", best_wer_line)
print(f"Using LM weight={BEST_LMWT}, word-insertion penalty={BEST_WIP}")


Best result: %WER 41.02 [ 2418 / 5894, 380 ins, 273 del, 1765 sub ] exp/tri3/decode_test/wer_24_1.0
Using LM weight=24, word-insertion penalty=1.0


## 2. Load Corpus (for Reference Text + Metadata)

Loaded in the same sorted-glob order `train_kaldi.ipynb` uses -- Kaldi's
utterance IDs encode each utterance's row position in this same
concatenation, and only resolve correctly if rebuilt identically. This
same order is also what `evaluate_elevenlabs.ipynb` uses, which is why
both notebooks' `corpus_index` values line up for `compare.ipynb`.

In [2]:
def load_all_shards(corpus_dir):
    files = sorted(Path(corpus_dir).glob("*.parquet"))
    dfs = [pd.read_parquet(f) for f in files]
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


corpus_df = load_all_shards(CORPUS_DIR)
print(f"Loaded {len(corpus_df):,} utterances from {CORPUS_DIR}")


Loaded 90 utterances from C:\Users\windows 10\Documents\Sugbodoc\speech model\data\bisaya_audio


## 3. Collect Predictions

Kaldi utterance IDs are `{speaker_id}-{corpus_index:06d}`. `reference` is
pulled from `corpus_df`'s raw `transcript` column (untouched casing/
punctuation), not Kaldi's own training-time-normalized `text` file --
matching what `evaluate_elevenlabs.ipynb` treats as ground truth, so the
two notebooks' metrics are comparable.

In [3]:
def load_kaldi_text(path):
    text_by_id = {}
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        parts = line.split(" ", 1)
        text_by_id[parts[0]] = parts[1] if len(parts) > 1 else ""
    return text_by_id


hyp_by_id = load_kaldi_text(HYP_PATH)

records = []
for utt_id, prediction in hyp_by_id.items():
    speaker_id, idx_str = utt_id.rsplit("-", 1)
    corpus_index = int(idx_str)
    corpus_row = corpus_df.loc[corpus_index]
    assert corpus_row["speaker_id"] == speaker_id, (
        f"utt_id {utt_id!r} maps to corpus row {corpus_index} with speaker "
        f"{corpus_row['speaker_id']!r}, expected {speaker_id!r}."
    )
    records.append({
        "id": utt_id,
        "speaker_id": speaker_id,
        "corpus_index": corpus_index,
        "reference": corpus_row["transcript"],
        "prediction": prediction,
    })

results_df = pd.DataFrame(records)
print(f"Loaded {len(results_df)} decoded test utterances.")
results_df.head()


Loaded 19 decoded test utterances.


,id,speaker_id,corpus_index,reference,prediction
0,CEB_001-000038,CEB_001,38,Kasagaran matulog ko mga alas dyes sa gabii. A...,kasagaran matulog koy mga alas diyes sa gabii ...
1,CEB_001-000039,CEB_001,39,"Kasagaran, mutulog ko nga gabii mga alas diyes...",kasagaran matulog ko nga gabii na nga alas diy...
2,CEB_001-000040,CEB_001,40,"Oo, kabalo ko mogamit og computer. Sa karon ng...",kakabalay kay gagamay pagkaon kayta isa karon ...
3,CEB_001-000041,CEB_001,41,"Oo, adunay koy opinyon bahin sa artificial int...",kaadunay koy pinya bahin sa arte opisyal binta...
4,CEB_001-000042,CEB_001,42,"Oo, mopalit ko online usahay tungod kay kini s...",oo ka kapalit kon naay nga usahay tungod kay k...


## 4. Raw WER/CER

Same raw metric definition as `evaluate_elevenlabs.ipynb`. Normalized
metrics are computed later in `compare.ipynb`.

In [4]:
from jiwer import process_words, process_characters

results = []
for _, row in results_df.iterrows():
    reference, prediction = row["reference"], row["prediction"]
    word_result = process_words(reference, prediction)
    char_result = process_characters(reference, prediction)

    results.append({
        "id": row["id"],
        "WER": word_result.wer,
        "WER_sub": word_result.substitutions,
        "WER_del": word_result.deletions,
        "WER_ins": word_result.insertions,
        "WER_hits": word_result.hits,
        "CER": char_result.cer,
        "CER_sub": char_result.substitutions,
        "CER_del": char_result.deletions,
        "CER_ins": char_result.insertions,
        "CER_hits": char_result.hits,
    })

metrics_df = pd.DataFrame(results)
metrics_df.head()


,id,WER,WER_sub,WER_del,WER_ins,WER_hits,CER,CER_sub,CER_del,CER_ins,CER_hits
0,CEB_001-000038,0.380859,161,6,28,345,0.111111,135,108,98,2826
1,CEB_001-000039,0.473161,192,8,38,303,0.174707,247,136,153,2685
2,CEB_001-000040,0.622845,222,7,60,235,0.252081,400,187,170,2416
3,CEB_001-000041,0.710817,238,5,79,210,0.304509,490,149,266,2333
4,CEB_001-000042,0.605809,224,12,56,246,0.256073,407,168,184,2389


## 5. Assemble Results for Export

In [5]:
META_COLS = [
    "speaker_id", "language", "gender", "country", "mother_tongue",
    "dialect", "os", "device", "n_words", "age_band", "native_speaker",
    "proficiency",
]

analysis_df = results_df[["id", "corpus_index", "reference", "prediction"]].copy()
for col in META_COLS:
    analysis_df[col] = analysis_df["corpus_index"].map(corpus_df[col])

analysis_df = analysis_df.merge(
    metrics_df[["id", "WER", "WER_sub", "WER_del", "WER_ins", "WER_hits",
                 "CER", "CER_sub", "CER_del", "CER_ins", "CER_hits"]],
    on="id", how="left",
)
analysis_df.head()


,id,corpus_index,reference,prediction,speaker_id,language,gender,country,mother_tongue,dialect,...,WER,WER_sub,WER_del,WER_ins,WER_hits,CER,CER_sub,CER_del,CER_ins,CER_hits
0,CEB_001-000038,38,Kasagaran matulog ko mga alas dyes sa gabii. A...,kasagaran matulog koy mga alas diyes sa gabii ...,CEB_001,Cebuano,female,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),...,0.380859,161,6,28,345,0.111111,135,108,98,2826
1,CEB_001-000039,39,"Kasagaran, mutulog ko nga gabii mga alas diyes...",kasagaran matulog ko nga gabii na nga alas diy...,CEB_001,Cebuano,female,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),...,0.473161,192,8,38,303,0.174707,247,136,153,2685
2,CEB_001-000040,40,"Oo, kabalo ko mogamit og computer. Sa karon ng...",kakabalay kay gagamay pagkaon kayta isa karon ...,CEB_001,Cebuano,female,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),...,0.622845,222,7,60,235,0.252081,400,187,170,2416
3,CEB_001-000041,41,"Oo, adunay koy opinyon bahin sa artificial int...",kaadunay koy pinya bahin sa arte opisyal binta...,CEB_001,Cebuano,female,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),...,0.710817,238,5,79,210,0.304509,490,149,266,2333
4,CEB_001-000042,42,"Oo, mopalit ko online usahay tungod kay kini s...",oo ka kapalit kon naay nga usahay tungod kay k...,CEB_001,Cebuano,female,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),...,0.605809,224,12,56,246,0.256073,407,168,184,2389


## 6. Export for Comparison

Writes per-utterance reference/prediction, demographics, and raw WER/CER
to `output/kaldi/results.csv` -- `compare.ipynb` reads this alongside
`output/elevenlabs/results.csv`, applies normalization, and does the
cross-system comparison.

In [6]:
OUTPUT_DIR = RESULTS_DIR / "kaldi"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_PATH = OUTPUT_DIR / "results.csv"
analysis_df.to_csv(EXPORT_PATH, index=False)
print(f"Wrote {len(analysis_df)} rows to {EXPORT_PATH}")


Wrote 19 rows to C:\Users\windows 10\Documents\Sugbodoc\speech model\output\kaldi\results.csv
